# Week 6 Practice: CNN Foundations

CSCI1409 Deep Learning · based on Lecture 5

## Instructions

- Work in this notebook with Python and PyTorch. You can use Google Colab.
- Do not use AI tools for any task (ChatGPT, Claude, Gemini, Copilot, Google AI Overviews).
- Show your code and its output. Be ready to explain every step.
- Write short answers to the explanation questions in the **Your answer** cells.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Task 1: Convolution

A convolution slides a small kernel over the input. At each position it multiplies the matching entries, sums them and adds a bias. The same kernel is used at every position.

We use the example from the lecture:

$$X = \begin{bmatrix} 1 & 2 & 0 & 1 \\ 0 & 1 & 3 & 2 \\ 2 & 2 & 1 & 0 \\ 1 & 0 & 2 & 3 \end{bmatrix}, \qquad K = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}$$

### 1a) One convolution by hand

1. Compute all four entries of $Y$ on paper (stride 1, padding 0, bias 0). Write the calculation for each entry.
2. Check your answer with `F.conv2d`. It expects the input as `[B, C, H, W]` and the kernel as `[C_out, C_in, k, k]`, so use `.view(1, 1, 4, 4)` and `.view(1, 1, 3, 3)`.
3. What would $Y$ be with bias $b = 1$?

In [2]:
X = torch.tensor([[1., 2., 0., 1.],
                  [0., 1., 3., 2.],
                  [2., 2., 1., 0.],
                  [1., 0., 2., 3.]])

K = torch.tensor([[1., 0., -1.],
                  [1., 0., -1.],
                  [1., 0., -1.]])

# Your code here

Y = F.conv2d(
    X.view(1,1,4,4),
    K.view(1,1,3,3)
)

print(Y)

tensor([[[[-1.,  2.],
          [-3., -2.]]]])


**Your answer:**

y + 1 =[
    0, 3
    -2, -1
]

### 1b) Output size and parameters

From the lecture:

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1, \qquad \text{params} = C_{out}\,(C_{in}\,k^2 + 1)$$

| Case | Input | Filters | $k$ | $p$ | $s$ |
|---|---|---|---|---|---|
| A | 1×28×28 | 8 | 3 | 0 | 1 |
| B | 1×28×28 | 8 | 3 | 1 | 1 |
| C | 1×28×28 | 8 | 3 | 1 | 2 |
| D | 3×32×32 | 16 | 5 | 2 | 1 |

1. Use the formulas to compute the output shape and the number of parameters for each case.
2. Check with `nn.Conv2d`. Pass `torch.randn(1, C_in, H, H)` through the layer, print the output shape, and count the parameters with `sum(p.numel() for p in layer.parameters())`.
3. A, B and C have the same number of parameters. Why?

In [3]:
# Your code here

layer = nn.Conv2d(1,8,3, padding = 0, stride = 1)
x = torch.randn(1,1,28,28)
y = layer(x)


print("a:", y.shape)
print("prs:", sum(p.numel() for p in layer.parameters()))


layer = nn.Conv2d(1,8, 3, padding=1, stride=1)
x = torch.randn(1, 1, 28, 28)
y = layer(x)

print("a:", y.shape)
print("prs:", sum(p.numel() for p in layer.parameters()))


layer = nn.Conv2d(1, 8, 3, padding=1, stride=2)
x = torch.randn(1, 1, 28, 28)
y = layer(x)

print("c:", y.shape)
print("prs:", sum(p.numel() for p in layer.parameters()))


layer = nn.Conv2d(3, 16, 5, padding=2, stride=1)
x = torch.randn(1, 3, 32, 32)
y = layer(x)

print("d:", y.shape)
print("prs:", sum(p.numel() for p in layer.parameters()))

a: torch.Size([1, 8, 26, 26])
prs: 80
a: torch.Size([1, 8, 28, 28])
prs: 80
c: torch.Size([1, 8, 14, 14])
prs: 80
d: torch.Size([1, 16, 32, 32])
prs: 1216


**Your answer:**

## Task 2: ReLU, pooling and the full CNN

### 2a) ReLU and max pooling

$Y$ is your result from 1a. $P$ is the matrix from the max pooling slide:

$$P = \begin{bmatrix} 1 & 3 & 2 & 0 \\ 4 & 2 & 1 & 5 \\ 0 & 1 & 3 & 2 \\ 2 & 0 & 1 & 4 \end{bmatrix}$$

1. Predict $\text{ReLU}(Y)$ and 2×2 max pooling with stride 2 of $P$ by hand.
2. Check with `nn.ReLU()` and `nn.MaxPool2d(2)`.
3. A batch of shape B×8×28×28 goes through `nn.MaxPool2d(2)`. What is the output shape? Why does the number of channels not change?

In [4]:
P = torch.tensor([[1., 3., 2., 0.],
                  [4., 2., 1., 5.],
                  [0., 1., 3., 2.],
                  [2., 0., 1., 4.]])

# Your code here
Rel = torch.nn.ReLU()
maxpool = nn.MaxPool2d(2)

print("relu print : ", Rel(Y))
print("max pooling: ", maxpool(P.view(1, 1, 4, 4))) # [B, C, H, W]

x = torch.randn(1, 8, 28, 28)
print(maxpool(x).shape)

relu print :  tensor([[[[0., 2.],
          [0., 0.]]]])
max pooling:  tensor([[[[4., 5.],
          [2., 4.]]]])
torch.Size([1, 8, 14, 14])


**Your answer:**

### 2b) The lecture CNN

1. Build the CNN from Lecture 5 (slide "The CNN in PyTorch") with `nn.Sequential`. Pass `x = torch.randn(4, 1, 28, 28)` through it one layer at a time and print the shape after each layer. Hint: `for layer in model: x = layer(x)`.
2. Count the parameters of the whole model and of each layer. What is the flattened feature count? Which layer has the most parameters?
3. An MLP for the same task (Flatten, Linear(784, 128), ReLU, Linear(128, 10)) has 101,770 parameters. Why does the CNN need so many fewer?

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10)
)


x = torch.randn(4, 1, 28, 28)

for layer in model:
    x = layer(x)
    print(layer, "->", x.shape)

total = 0

for layer in model:
    params = sum(p.numel() for p in layer.parameters())
    print(layer, "parameters:", params)
    total += params

print("Total parameters:", total)

print("MLP parameters:", 101770)
print("CNN parameters:", total)

Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)) -> torch.Size([4, 8, 28, 28])
ReLU() -> torch.Size([4, 8, 28, 28])
MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False) -> torch.Size([4, 8, 14, 14])
Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)) -> torch.Size([4, 16, 14, 14])
ReLU() -> torch.Size([4, 16, 14, 14])
MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False) -> torch.Size([4, 16, 7, 7])
Flatten(start_dim=1, end_dim=-1) -> torch.Size([4, 784])
Linear(in_features=784, out_features=10, bias=True) -> torch.Size([4, 10])
Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)) parameters: 80
ReLU() parameters: 0
MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False) parameters: 0
Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)) parameters: 1168
ReLU() parameters: 0
MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False) parameters: 0
Flatten(start

**Your answer:**

## Stretch (optional): convolution with loops

Write `conv2d_manual(X, K)` with two `for` loops (stride 1, padding 0, no bias). At output position $(i, j)$ take the patch `X[i:i+k, j:j+k]`, multiply it with K element by element and sum. Check that you get the same $Y$ as in 1a.

In [ ]:
# Your code here